# 03 · Đánh giá và so sánh thuật toán

**Đồ án:** Phân cụm các quốc gia theo đặc điểm kinh tế vĩ mô và phát hiện các nền kinh tế khác biệt bằng DBSCAN.

**Phụ trách:** Khoa · **Trạng thái:** Đã triển khai mã nguồn, chờ/đọc kết quả DBSCAN từ notebook 02 để chạy thực nghiệm cuối.

Notebook đánh giá DBSCAN và ba thuật toán đối chứng trên đúng cùng dữ liệu chuẩn hóa: K-Means, Agglomerative Clustering và Gaussian Mixture Model (GMM). Ngoài bảng chỉ số, notebook tạo PCA 2D, hồ sơ cụm ở đơn vị gốc, quan sát đại diện và bảng phân tích noise.

[README dự án](../../README.md) · [Hướng dẫn dữ liệu](../../du_lieu/README.md) · [Quy ước kết quả](../../ket_qua_chung/README.md)


## 0. Đầu vào bắt buộc và vị trí đặt file

Tất cả đường dẫn dưới đây được tính từ **gốc repository DBSCAN**. Khi nhận file bàn giao, giữ nguyên tên và đặt đúng vị trí; **không đặt vào thư mục `Khoa/notebook/`**.

| File đầu vào | Người bàn giao | Vai trò | Trạng thái trước khi chạy |
| --- | --- | --- | --- |
| `du_lieu/du_lieu_sach/wdi_clean.csv` | Đức | Hồ sơ cụm theo đơn vị kinh tế gốc | Bắt buộc |
| `du_lieu/du_lieu_chuan_hoa/wdi_scaled.csv` | Đức | Ma trận 8 đặc trưng dùng chung cho mọi thuật toán | Bắt buộc |
| `du_lieu/du_lieu_chuan_hoa/preprocessing_metadata.json` | Đức | Thứ tự đặc trưng, biến đổi, phạm vi chuẩn hóa và checksum | Bắt buộc |
| `ket_qua_chung/dbscan/dbscan_labels.csv` | Khang | Nhãn `cluster` và loại điểm `core/border/noise` | Bắt buộc |
| `ket_qua_chung/dbscan/tham_so_da_chon.json` | Khang | `eps`, `min_samples`, metric và phạm vi fit panel/theo năm | Bắt buộc |

Cấu trúc thư mục cần có trước khi **Run All**:

```text
DBSCAN/
├── du_lieu/
│   ├── du_lieu_sach/wdi_clean.csv
│   └── du_lieu_chuan_hoa/
│       ├── wdi_scaled.csv
│       └── preprocessing_metadata.json
└── ket_qua_chung/dbscan/
    ├── dbscan_labels.csv
    └── tham_so_da_chon.json
```

Notebook tự tìm gốc repo dù kernel khởi động ở gốc dự án hay trong `Khoa/notebook`. Cell cấu hình ngay bên dưới hiển thị bảng **CÓ/THIẾU** cho cả 5 file; nếu thiếu, notebook dừng và liệt kê chính xác đường dẫn cần bổ sung.


## 1. Mục tiêu và nguyên tắc đánh giá

Ba câu hỏi chính:

1. Các cụm có tách biệt tốt không theo Silhouette, Davies–Bouldin và Calinski–Harabasz?
2. Kết quả đạt được trên bao nhiêu quan sát, đặc biệt khi DBSCAN loại noise?
3. Hồ sơ cụm và các điểm noise khác nhau ở chỉ tiêu kinh tế nào?

Nguyên tắc công bằng:

- Ghép bảng duy nhất bằng khóa country + year, kiểm tra quan hệ một-một.
- Mọi thuật toán dùng cùng 8 đặc trưng và cùng dữ liệu đã biến đổi/chuẩn hóa.
- Đối chứng được fit trên toàn bộ mẫu của từng phạm vi chạy; không fit lại trên tập non-noise.
- Đánh giá chính của DBSCAN loại noise và luôn báo coverage; phép so sánh bổ sung dùng cùng tập non-noise của DBSCAN cho cả bốn thuật toán.
- Chỉ tính chỉ số khi 2 ≤ số nhãn < số mẫu; trường hợp khác giữ NaN và ghi lý do.
- PCA chỉ dùng để trực quan hóa nhãn đã fit trong không gian 8 chiều.


## 2. Chuẩn bị môi trường, đường dẫn và cấu hình

Nếu chạy từ thư mục notebook hoặc từ gốc repo, cell dưới đây đều tự tìm đúng gốc dự án. Có thể đặt EVALUATION_MODE thành "panel" hoặc "per_year" để ghi đè chế độ tự nhận diện từ tham_so_da_chon.json.


In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
MAX_K = 10
KMEANS_N_INIT = 50
GMM_N_INIT = 5
EVALUATION_MODE = "auto"  # "auto", "panel" hoặc "per_year"
REPRESENTATIVES_PER_CLUSTER = 5

KEY_COLUMNS = ["country", "year"]
FEATURE_LABELS = {
    "NY.GDP.PCAP.KD": "GDP/người (USD cố định 2015)",
    "NY.GDP.MKTP.KD.ZG": "Tăng trưởng GDP (%/năm)",
    "FP.CPI.TOTL.ZG": "Lạm phát CPI (%/năm)",
    "SL.UEM.TOTL.ZS": "Thất nghiệp (% LLLĐ)",
    "NE.TRD.GNFS.ZS": "Thương mại (% GDP)",
    "NE.GDI.TOTL.ZS": "Tích lũy vốn (% GDP)",
    "SP.URB.TOTL.IN.ZS": "Dân số đô thị (%)",
    "NE.CON.GOVT.ZS": "Chi tiêu chính phủ (% GDP)",
}

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").is_file()
            and (candidate / "du_lieu").is_dir()
            and (candidate / "Khoa").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Không tìm thấy gốc repo DBSCAN từ thư mục hiện tại.")

ROOT = find_repo_root()
PATHS = {
    "clean": ROOT / "du_lieu/du_lieu_sach/wdi_clean.csv",
    "scaled": ROOT / "du_lieu/du_lieu_chuan_hoa/wdi_scaled.csv",
    "metadata": ROOT / "du_lieu/du_lieu_chuan_hoa/preprocessing_metadata.json",
    "dbscan_labels": ROOT / "ket_qua_chung/dbscan/dbscan_labels.csv",
    "dbscan_params": ROOT / "ket_qua_chung/dbscan/tham_so_da_chon.json",
    "evaluation": ROOT / "ket_qua_chung/danh_gia",
    "profiles": ROOT / "ket_qua_chung/ho_so_cum",
    "noise": ROOT / "ket_qua_chung/phan_tich_nhieu",
    "figures": ROOT / "Khoa/bieu_do",
    "report_figures": ROOT / "ket_qua_chung/bieu_do_bao_cao",
}
for name in ["evaluation", "profiles", "noise", "figures", "report_figures"]:
    PATHS[name].mkdir(parents=True, exist_ok=True)

INPUT_MANIFEST = [
    {"key": "clean", "owner": "Đức", "purpose": "Dữ liệu sạch ở đơn vị gốc"},
    {"key": "scaled", "owner": "Đức", "purpose": "Ma trận đặc trưng chuẩn hóa"},
    {"key": "metadata", "owner": "Đức", "purpose": "Metadata tiền xử lý và thứ tự đặc trưng"},
    {"key": "dbscan_labels", "owner": "Khang", "purpose": "Nhãn cụm và loại điểm DBSCAN"},
    {"key": "dbscan_params", "owner": "Khang", "purpose": "Tham số và phạm vi fit DBSCAN"},
]
input_status = pd.DataFrame(
    [
        {
            "người_bàn_giao": item["owner"],
            "đường_dẫn_từ_gốc_repo": str(PATHS[item["key"]].relative_to(ROOT)),
            "vai_trò": item["purpose"],
            "trạng_thái": "CÓ" if PATHS[item["key"]].is_file() else "THIẾU",
        }
        for item in INPUT_MANIFEST
    ]
)

print(f"Gốc repo: {ROOT}")
print(f"scikit-learn: {sklearn.__version__} | pandas: {pd.__version__} | numpy: {np.__version__}")
display(input_status)
if input_status["trạng_thái"].eq("THIẾU").any():
    print("Chưa thể chạy phân tích: đặt các file THIẾU vào đúng đường dẫn hiển thị ở trên.")


In [ ]:
required_inputs = [PATHS[item["key"]] for item in INPUT_MANIFEST]
missing_inputs = [str(path.relative_to(ROOT)) for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "Thiếu đầu vào bàn giao nên notebook dừng để tránh tạo kết quả giả:\n- "
        + "\n- ".join(missing_inputs)
        + "\nHãy chạy notebook 01 và 02 theo đúng thứ tự, sau đó Run All notebook 03."
    )

clean = pd.read_csv(PATHS["clean"])
scaled = pd.read_csv(PATHS["scaled"])
dbscan_labels = pd.read_csv(PATHS["dbscan_labels"])
with PATHS["metadata"].open(encoding="utf-8") as f:
    preprocessing_metadata = json.load(f)
with PATHS["dbscan_params"].open(encoding="utf-8") as f:
    dbscan_params = json.load(f)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

expected_hashes = {
    "clean": preprocessing_metadata.get("outputs", {}).get("clean_sha256"),
    "scaled": preprocessing_metadata.get("outputs", {}).get("scaled_sha256"),
}
checksum_audit = {}
for dataset_name, expected_hash in expected_hashes.items():
    actual_hash = sha256_file(PATHS[dataset_name])
    checksum_audit[dataset_name] = {
        "expected": expected_hash,
        "actual": actual_hash,
        "match": expected_hash == actual_hash if expected_hash else None,
    }
    if expected_hash and actual_hash != expected_hash:
        warnings.warn(
            f"Checksum {dataset_name} không khớp metadata; tiếp tục vì các kiểm tra "
            "schema, khóa và số dòng vẫn được thực hiện. Cần đồng bộ lại metadata ở notebook 01."
        )

feature_columns = preprocessing_metadata["feature_columns"]
missing_feature_labels = sorted(set(feature_columns) - set(FEATURE_LABELS))
if missing_feature_labels:
    raise ValueError(f"Chưa khai báo nhãn cho đặc trưng: {missing_feature_labels}")

def assert_unique_keys(df: pd.DataFrame, name: str) -> None:
    if df[KEY_COLUMNS].isna().any().any():
        raise ValueError(f"{name}: khóa country-year có giá trị thiếu.")
    duplicates = df.duplicated(KEY_COLUMNS, keep=False)
    if duplicates.any():
        sample = df.loc[duplicates, KEY_COLUMNS].head().to_dict("records")
        raise ValueError(f"{name}: khóa country-year bị trùng, ví dụ {sample}")

required_clean = set(KEY_COLUMNS + ["country_name", "is_aggregate"] + feature_columns)
required_scaled = set(KEY_COLUMNS + feature_columns)
required_labels = set(KEY_COLUMNS + ["cluster", "point_type"])
for df, required, name in [
    (clean, required_clean, "wdi_clean.csv"),
    (scaled, required_scaled, "wdi_scaled.csv"),
    (dbscan_labels, required_labels, "dbscan_labels.csv"),
]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name}: thiếu cột {sorted(missing)}")
    assert_unique_keys(df, name)

clean["year"] = pd.to_numeric(clean["year"], errors="raise").astype(int)
scaled["year"] = pd.to_numeric(scaled["year"], errors="raise").astype(int)
dbscan_labels["year"] = pd.to_numeric(dbscan_labels["year"], errors="raise").astype(int)
dbscan_labels["cluster"] = pd.to_numeric(dbscan_labels["cluster"], errors="raise").astype(int)
dbscan_labels["point_type"] = dbscan_labels["point_type"].astype(str).str.lower()

def key_index(df: pd.DataFrame) -> pd.MultiIndex:
    return pd.MultiIndex.from_frame(df[KEY_COLUMNS].sort_values(KEY_COLUMNS).reset_index(drop=True))

scaled_keys = key_index(scaled)
clean_keys = key_index(clean)
label_keys = key_index(dbscan_labels)
if not scaled_keys.equals(clean_keys):
    raise ValueError("Tập khóa dữ liệu sạch không khớp dữ liệu chuẩn hóa.")
if not scaled_keys.equals(label_keys):
    only_scaled = scaled_keys.difference(label_keys)
    only_labels = label_keys.difference(scaled_keys)
    raise ValueError(
        f"Nhãn DBSCAN không khớp tập mẫu: thiếu {len(only_scaled)}, thừa {len(only_labels)} khóa."
    )

valid_point_types = {"core", "border", "noise"}
unknown_types = set(dbscan_labels["point_type"]) - valid_point_types
if unknown_types:
    raise ValueError(f"point_type không hợp lệ: {sorted(unknown_types)}")
noise_consistent = (dbscan_labels["cluster"].eq(-1) == dbscan_labels["point_type"].eq("noise")).all()
if not noise_consistent:
    raise ValueError("Quy ước cluster=-1 và point_type=noise không nhất quán.")
if (clean["is_aggregate"] != 0).any():
    raise ValueError("Dữ liệu sạch vẫn còn nhóm tổng hợp.")
if not np.isfinite(scaled[feature_columns].to_numpy(dtype=float)).all():
    raise ValueError("Ma trận đặc trưng chuẩn hóa còn NaN hoặc vô cực.")

analysis = (
    scaled[KEY_COLUMNS + feature_columns]
    .merge(dbscan_labels[KEY_COLUMNS + ["cluster", "point_type"]], on=KEY_COLUMNS, validate="one_to_one")
    .merge(clean[KEY_COLUMNS + ["country_name"]], on=KEY_COLUMNS, validate="one_to_one")
    .sort_values(KEY_COLUMNS)
    .reset_index(drop=True)
)

def iter_json_items(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            yield from iter_json_items(value, f"{prefix}.{key}" if prefix else str(key))
    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            yield from iter_json_items(value, f"{prefix}[{i}]")
    else:
        yield prefix.lower(), obj

def infer_evaluation_mode(params: dict) -> tuple[str, str]:
    if EVALUATION_MODE in {"panel", "per_year"}:
        return EVALUATION_MODE, "ghi đè thủ công trong notebook"
    if EVALUATION_MODE != "auto":
        raise ValueError("EVALUATION_MODE phải là auto, panel hoặc per_year.")

    items = list(iter_json_items(params))
    explicit_keys = (
        "fit_scope", "fitting_scope", "evaluation_scope", "clustering_scope",
        "scope.mode", "study_scope.mode", "mode",
    )
    explicit_values = [
        str(value).lower()
        for key, value in items
        if any(key == token or key.endswith("." + token) for token in explicit_keys)
    ]
    all_keys = " ".join(key for key, _ in items)
    all_values = " ".join(str(value).lower() for _, value in items)

    yearly_markers = ("per_year", "yearly", "từng năm", "theo năm", "fit riêng")
    panel_markers = ("global_panel", "panel", "gộp panel", "toàn panel")
    if any(marker in " ".join(explicit_values) for marker in yearly_markers):
        return "per_year", "tham_so_da_chon.json khai báo fit theo năm"
    if any(marker in " ".join(explicit_values) for marker in panel_markers):
        return "panel", "tham_so_da_chon.json khai báo fit panel"
    if any(token in all_keys for token in ("params_by_year", "parameters_by_year", "year_configs", "yearly_params")):
        return "per_year", "tham số DBSCAN được lưu riêng theo năm"
    if any(marker in all_values for marker in yearly_markers):
        return "per_year", "mô tả tham số có dấu hiệu fit theo năm"
    if any(marker in all_values for marker in panel_markers):
        return "panel", "mô tả tham số có dấu hiệu fit panel"
    return "panel", "không thấy khai báo rõ; dùng mặc định panel và cần xác nhận với notebook 02"

evaluation_mode, mode_reason = infer_evaluation_mode(dbscan_params)
analysis["evaluation_scope"] = (
    analysis["year"].astype(str)
    if evaluation_mode == "per_year"
    else f"{analysis['year'].min()}-{analysis['year'].max()}_panel"
)

input_audit = pd.DataFrame(
    {
        "table": ["clean", "scaled", "dbscan_labels"],
        "rows": [len(clean), len(scaled), len(dbscan_labels)],
        "unique_country_year": [
            clean[KEY_COLUMNS].drop_duplicates().shape[0],
            scaled[KEY_COLUMNS].drop_duplicates().shape[0],
            dbscan_labels[KEY_COLUMNS].drop_duplicates().shape[0],
        ],
        "countries": [clean["country"].nunique(), scaled["country"].nunique(), dbscan_labels["country"].nunique()],
        "year_min": [clean["year"].min(), scaled["year"].min(), dbscan_labels["year"].min()],
        "year_max": [clean["year"].max(), scaled["year"].max(), dbscan_labels["year"].max()],
    }
)
print(f"Chế độ đánh giá: {evaluation_mode} ({mode_reason})")
display(input_audit)
display(dbscan_labels["point_type"].value_counts().rename_axis("point_type").to_frame("n"))


## 3–5. Triển khai các thuật toán đối chứng

Việc chọn mô hình được thực hiện độc lập trong mỗi phạm vi fit của DBSCAN:

- **K-Means:** thử k = 2..10, k-means++, n_init=50, chọn Silhouette cao nhất.
- **Agglomerative:** thử cùng dải k, Ward linkage với khoảng cách Euclid, chọn Silhouette cao nhất.
- **GMM:** thử k = 2..10 và bốn dạng covariance (full, tied, diag, spherical), chọn BIC thấp nhất.

Các nhãn đối chứng luôn được fit trên toàn bộ mẫu của phạm vi tương ứng. Dải tìm kiếm, seed và cấu hình thắng được xuất riêng để có thể truy vết.


In [ ]:
def safe_internal_metrics(X: np.ndarray, labels: np.ndarray) -> dict:
    labels = np.asarray(labels)
    n_samples = len(labels)
    n_clusters = len(np.unique(labels))
    if n_samples < 3:
        reason = f"Không hợp lệ: chỉ có {n_samples} mẫu."
        return {"silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan, "metric_note": reason}
    if not (2 <= n_clusters < n_samples):
        reason = f"Không hợp lệ: {n_clusters} nhãn trên {n_samples} mẫu."
        return {"silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan, "metric_note": reason}
    return {
        "silhouette": float(silhouette_score(X, labels)),
        "davies_bouldin": float(davies_bouldin_score(X, labels)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
        "metric_note": "Hợp lệ",
    }

def fit_baselines_for_scope(scope_df: pd.DataFrame) -> tuple[dict, list[dict], dict]:
    X = scope_df[feature_columns].to_numpy(dtype=float)
    scope_name = str(scope_df["evaluation_scope"].iloc[0])
    max_k = min(MAX_K, len(scope_df) - 1)
    candidate_k = list(range(2, max_k + 1))
    if not candidate_k:
        raise ValueError(f"{scope_name}: không đủ mẫu để fit thuật toán đối chứng.")

    grid_rows = []

    kmeans_candidates = []
    for k in candidate_k:
        model = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=KMEANS_N_INIT,
            random_state=RANDOM_STATE,
        )
        labels = model.fit_predict(X)
        score = float(silhouette_score(X, labels))
        kmeans_candidates.append((score, k, labels, model))
        grid_rows.append({
            "evaluation_scope": scope_name, "algorithm": "K-Means", "n_clusters": k,
            "covariance_type": pd.NA, "selection_metric": "silhouette",
            "selection_value": score,
        })
    best_kmeans = max(kmeans_candidates, key=lambda row: row[0])

    agg_candidates = []
    for k in candidate_k:
        model = AgglomerativeClustering(n_clusters=k, metric="euclidean", linkage="ward")
        labels = model.fit_predict(X)
        score = float(silhouette_score(X, labels))
        agg_candidates.append((score, k, labels, model))
        grid_rows.append({
            "evaluation_scope": scope_name, "algorithm": "Agglomerative", "n_clusters": k,
            "covariance_type": pd.NA, "selection_metric": "silhouette",
            "selection_value": score,
        })
    best_agg = max(agg_candidates, key=lambda row: row[0])

    gmm_candidates = []
    for covariance_type in ["full", "tied", "diag", "spherical"]:
        for k in candidate_k:
            model = GaussianMixture(
                n_components=k,
                covariance_type=covariance_type,
                n_init=GMM_N_INIT,
                reg_covar=1e-6,
                random_state=RANDOM_STATE,
            )
            labels = model.fit_predict(X)
            bic = float(model.bic(X))
            gmm_candidates.append((bic, k, covariance_type, labels, model))
            grid_rows.append({
                "evaluation_scope": scope_name, "algorithm": "GMM", "n_clusters": k,
                "covariance_type": covariance_type, "selection_metric": "bic",
                "selection_value": bic,
            })
    best_gmm = min(gmm_candidates, key=lambda row: row[0])

    labels_by_algorithm = {
        "K-Means": best_kmeans[2],
        "Agglomerative": best_agg[2],
        "GMM": best_gmm[3],
    }
    selected_config = {
        "K-Means": {
            "n_clusters": int(best_kmeans[1]), "init": "k-means++",
            "n_init": KMEANS_N_INIT, "random_state": RANDOM_STATE,
            "selection": "maximum silhouette on full fit sample",
        },
        "Agglomerative": {
            "n_clusters": int(best_agg[1]), "linkage": "ward", "metric": "euclidean",
            "selection": "maximum silhouette on full fit sample",
        },
        "GMM": {
            "n_components": int(best_gmm[1]), "covariance_type": best_gmm[2],
            "n_init": GMM_N_INIT, "reg_covar": 1e-6, "random_state": RANDOM_STATE,
            "selection": "minimum BIC on full fit sample",
        },
    }
    return labels_by_algorithm, grid_rows, selected_config


In [ ]:
algorithm_columns = {
    "DBSCAN": "cluster",
    "K-Means": "kmeans_cluster",
    "Agglomerative": "agglomerative_cluster",
    "GMM": "gmm_cluster",
}
column_for_algorithm = {
    "K-Means": "kmeans_cluster",
    "Agglomerative": "agglomerative_cluster",
    "GMM": "gmm_cluster",
}

analysis_with_labels = analysis.copy()
for column in column_for_algorithm.values():
    analysis_with_labels[column] = pd.NA

model_selection_rows = []
selected_configs = {}

for scope, scope_indices in analysis_with_labels.groupby("evaluation_scope", sort=True).groups.items():
    scope_df = analysis_with_labels.loc[scope_indices].copy()
    fitted_labels, grid_rows, configs = fit_baselines_for_scope(scope_df)
    for algorithm, labels in fitted_labels.items():
        analysis_with_labels.loc[scope_indices, column_for_algorithm[algorithm]] = labels
    model_selection_rows.extend(grid_rows)
    selected_configs[str(scope)] = configs

for column in column_for_algorithm.values():
    analysis_with_labels[column] = analysis_with_labels[column].astype(int)

model_selection = pd.DataFrame(model_selection_rows)
baseline_labels_output = analysis_with_labels[
    KEY_COLUMNS + ["evaluation_scope", "cluster", "point_type"] + list(column_for_algorithm.values())
].copy()

model_selection.to_csv(PATHS["evaluation"] / "lua_chon_mo_hinh_doi_chung.csv", index=False)
baseline_labels_output.to_csv(PATHS["evaluation"] / "nhan_thuat_toan_doi_chung.csv", index=False)

baseline_config_payload = {
    "evaluation_mode": evaluation_mode,
    "mode_reason": mode_reason,
    "feature_columns": feature_columns,
    "input_checksum_audit": checksum_audit,
    "random_state": RANDOM_STATE,
    "max_k": MAX_K,
    "selection_by_scope": selected_configs,
    "library_versions": {
        "python_note": "Xem kernel notebook",
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
    },
}
with (PATHS["evaluation"] / "cau_hinh_thuat_toan_doi_chung.json").open("w", encoding="utf-8") as f:
    json.dump(baseline_config_payload, f, ensure_ascii=False, indent=2)

selected_config_table = pd.DataFrame(
    [
        {"evaluation_scope": scope, "algorithm": algorithm, **config}
        for scope, configs in selected_configs.items()
        for algorithm, config in configs.items()
    ]
)
display(selected_config_table.head(12))


## 6–9. Các chỉ số và bảng so sánh

- **Silhouette:** cao hơn tốt hơn, trong khoảng [-1, 1].
- **Davies–Bouldin:** thấp hơn tốt hơn.
- **Calinski–Harabasz:** cao hơn tốt hơn khi so trên cùng tập mẫu.

Mỗi thuật toán có hai dòng đánh giá cho từng phạm vi:

- algorithm_policy: DBSCAN loại noise; ba đối chứng dùng toàn bộ mẫu.
- dbscan_non_noise_common_subset: cả bốn thuật toán cùng được chấm trên tập non-noise của DBSCAN, nhưng giữ nguyên nhãn đã fit trên toàn mẫu.

Nhờ vậy, điểm số không bị trình bày tách rời khỏi n_evaluated, coverage và tỷ lệ noise tham chiếu.


In [ ]:
comparison_rows = []

for scope, scope_df in analysis_with_labels.groupby("evaluation_scope", sort=True):
    X = scope_df[feature_columns].to_numpy(dtype=float)
    dbscan_non_noise = scope_df["cluster"].to_numpy() != -1
    n_input = len(scope_df)
    year_start = int(scope_df["year"].min())
    year_end = int(scope_df["year"].max())
    dbscan_noise_rate = float((~dbscan_non_noise).mean())

    for algorithm, label_column in algorithm_columns.items():
        labels_full = scope_df[label_column].to_numpy(dtype=int)
        masks = {
            "algorithm_policy": dbscan_non_noise if algorithm == "DBSCAN" else np.ones(n_input, dtype=bool),
            "dbscan_non_noise_common_subset": dbscan_non_noise,
        }
        for evaluation_set, mask in masks.items():
            X_eval = X[mask]
            labels_eval = labels_full[mask]
            metrics = safe_internal_metrics(X_eval, labels_eval)
            config = dbscan_params if algorithm == "DBSCAN" else selected_configs[str(scope)][algorithm]
            comparison_rows.append({
                "run_id": f"{scope}__{algorithm.lower().replace(' ', '_')}__{evaluation_set}",
                "algorithm": algorithm,
                "fit_scope": evaluation_mode,
                "evaluation_scope": scope,
                "evaluation_set": evaluation_set,
                "year_start": year_start,
                "year_end": year_end,
                "n_input": n_input,
                "n_evaluated": int(mask.sum()),
                "coverage": float(mask.mean()),
                "n_clusters": int(len(np.unique(labels_eval))),
                "algorithm_noise_rate": dbscan_noise_rate if algorithm == "DBSCAN" else 0.0,
                "dbscan_reference_noise_rate": dbscan_noise_rate,
                "silhouette": metrics["silhouette"],
                "davies_bouldin": metrics["davies_bouldin"],
                "calinski_harabasz": metrics["calinski_harabasz"],
                "metric_note": metrics["metric_note"],
                "configuration": json.dumps(config, ensure_ascii=False, sort_keys=True),
            })

comparison = pd.DataFrame(comparison_rows).sort_values(
    ["evaluation_scope", "evaluation_set", "algorithm"]
).reset_index(drop=True)
comparison_path = PATHS["evaluation"] / "so_sanh_thuat_toan.csv"
comparison.to_csv(comparison_path, index=False)

main_summary = (
    comparison.query("evaluation_set == 'algorithm_policy'")
    .groupby("algorithm", as_index=False)
    .agg(
        scopes=("evaluation_scope", "nunique"),
        n_input_mean=("n_input", "mean"),
        n_evaluated_mean=("n_evaluated", "mean"),
        coverage_mean=("coverage", "mean"),
        clusters_mean=("n_clusters", "mean"),
        silhouette_mean=("silhouette", "mean"),
        davies_bouldin_mean=("davies_bouldin", "mean"),
        calinski_harabasz_mean=("calinski_harabasz", "mean"),
    )
)
display(main_summary.round(4))
print(f"Đã lưu: {comparison_path.relative_to(ROOT)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=main_summary, x="algorithm", y="silhouette_mean", ax=axes[0], color="#4C78A8")
axes[0].set(title="Silhouette trung bình theo chính sách thuật toán", xlabel="", ylabel="Silhouette (cao hơn tốt hơn)")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(data=main_summary, x="algorithm", y="coverage_mean", ax=axes[1], color="#F58518")
axes[1].set(title="Độ phủ trung bình", xlabel="", ylabel="Coverage")
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis="x", rotation=20)
fig.suptitle("So sánh chất lượng cụm và độ phủ", fontweight="bold")
fig.tight_layout()
comparison_figure = PATHS["figures"] / "02_so_sanh_chi_so_va_do_phu.png"
fig.savefig(comparison_figure, dpi=180, bbox_inches="tight")
plt.show()
shutil.copy2(comparison_figure, PATHS["report_figures"] / comparison_figure.name)


## 10. Trực quan hóa PCA 2D

PCA được fit một lần trên toàn bộ ma trận chuẩn hóa để bốn biểu đồ dùng cùng hệ trục. Nếu DBSCAN fit riêng từng năm, hình chọn năm mới nhất; bảng kết quả CSV vẫn giữ đầy đủ mọi năm. Noise được vẽ bằng dấu × màu đen.


In [ ]:
X_all = analysis_with_labels[feature_columns].to_numpy(dtype=float)
pca = PCA(n_components=2)
coordinates = pca.fit_transform(X_all)
pca_df = analysis_with_labels.copy()
pca_df["PC1"] = coordinates[:, 0]
pca_df["PC2"] = coordinates[:, 1]

if evaluation_mode == "per_year":
    visual_scope = sorted(pca_df["evaluation_scope"].unique(), key=lambda x: int(x))[-1]
else:
    visual_scope = pca_df["evaluation_scope"].iloc[0]
plot_df = pca_df.loc[pca_df["evaluation_scope"].eq(visual_scope)].copy()

fig, axes = plt.subplots(2, 2, figsize=(15, 11), sharex=True, sharey=True)
for ax, (algorithm, label_column) in zip(axes.flat, algorithm_columns.items()):
    labels = plot_df[label_column].to_numpy(dtype=int)
    unique_labels = sorted(np.unique(labels))
    non_noise_labels = [label for label in unique_labels if label != -1]
    palette = sns.color_palette("tab10", n_colors=max(1, len(non_noise_labels)))
    color_map = {label: palette[i % len(palette)] for i, label in enumerate(non_noise_labels)}

    for label in non_noise_labels:
        mask = labels == label
        ax.scatter(
            plot_df.loc[mask, "PC1"], plot_df.loc[mask, "PC2"],
            s=28, alpha=0.72, color=color_map[label], label=f"Cụm {label}",
            linewidths=0,
        )
    if -1 in unique_labels:
        mask = labels == -1
        ax.scatter(
            plot_df.loc[mask, "PC1"], plot_df.loc[mask, "PC2"],
            s=44, alpha=0.9, color="black", marker="x", label="Noise",
        )
    ax.set_title(algorithm, fontweight="bold")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax.legend(fontsize=8, ncol=2, frameon=True)

fig.suptitle(
    f"PCA 2D – phạm vi {visual_scope} | phương sai giải thích cộng dồn: "
    f"{pca.explained_variance_ratio_[:2].sum():.1%}",
    fontsize=14, fontweight="bold",
)
fig.tight_layout()
pca_figure = PATHS["figures"] / "01_pca_so_sanh_thuat_toan.png"
fig.savefig(pca_figure, dpi=180, bbox_inches="tight")
plt.show()
shutil.copy2(pca_figure, PATHS["report_figures"] / pca_figure.name)

pca_metadata = {
    "fit_scope": "all rows in wdi_scaled.csv",
    "visualized_scope": str(visual_scope),
    "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    "explained_variance_ratio_sum_2d": float(pca.explained_variance_ratio_[:2].sum()),
    "feature_columns": feature_columns,
}
with (PATHS["evaluation"] / "pca_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(pca_metadata, f, ensure_ascii=False, indent=2)


## 11. Hồ sơ kinh tế của cụm DBSCAN

Nhãn DBSCAN được ghép lại dữ liệu sạch theo country + year. Với mỗi cụm, notebook xuất quy mô, số nền kinh tế, khoảng năm và thống kê mean/median/std/q25/q75 ở đơn vị gốc. Quan sát đại diện là các hàng gần trung vị cụm nhất trong không gian chuẩn hóa; đây là đại diện hình học, không phải xếp hạng chất lượng nền kinh tế.


In [ ]:
clean_original = clean[KEY_COLUMNS + ["country_name"] + feature_columns].copy()
profile_data = (
    analysis_with_labels[
        KEY_COLUMNS + ["evaluation_scope", "cluster", "point_type"] + feature_columns
    ]
    .merge(clean_original, on=KEY_COLUMNS, how="left", validate="one_to_one", suffixes=("_scaled", "_original"))
)
cluster_data = profile_data.loc[profile_data["cluster"] != -1].copy()
if cluster_data.empty:
    raise ValueError("DBSCAN không có điểm non-noise; không thể lập hồ sơ cụm.")

profile_rows = []
representative_rows = []

for (scope, cluster), group in cluster_data.groupby(["evaluation_scope", "cluster"], sort=True):
    base = {
        "evaluation_scope": scope,
        "cluster": int(cluster),
        "n_observations": len(group),
        "n_economies": group["country"].nunique(),
        "year_min": int(group["year"].min()),
        "year_max": int(group["year"].max()),
    }
    for feature in feature_columns:
        values = group[f"{feature}_original"]
        profile_rows.append({
            **base,
            "feature": feature,
            "feature_name": FEATURE_LABELS[feature],
            "mean": float(values.mean()),
            "median": float(values.median()),
            "std": float(values.std(ddof=1)) if len(values) > 1 else np.nan,
            "q25": float(values.quantile(0.25)),
            "q75": float(values.quantile(0.75)),
        })

    scaled_cols = [f"{feature}_scaled" for feature in feature_columns]
    center = group[scaled_cols].median().to_numpy(dtype=float)
    distances = np.linalg.norm(group[scaled_cols].to_numpy(dtype=float) - center, axis=1)
    representatives = group.assign(distance_to_cluster_median=distances).nsmallest(
        REPRESENTATIVES_PER_CLUSTER, "distance_to_cluster_median"
    )
    for rank, (_, observation) in enumerate(representatives.iterrows(), start=1):
        representative_rows.append({
            "evaluation_scope": scope,
            "cluster": int(cluster),
            "rank": rank,
            "country": observation["country"],
            "country_name": observation["country_name"],
            "year": int(observation["year"]),
            "distance_to_cluster_median": float(observation["distance_to_cluster_median"]),
        })

profile_wide = pd.DataFrame(
    [
        {
            "evaluation_scope": scope,
            "cluster": int(cluster),
            "n_observations": len(group),
            "n_economies": group["country"].nunique(),
            "year_min": int(group["year"].min()),
            "year_max": int(group["year"].max()),
            **{
                f"{feature}__{stat}": float(getattr(group[f"{feature}_original"], stat)())
                if stat in {"mean", "median"} else
                (float(group[f"{feature}_original"].std(ddof=1)) if len(group) > 1 else np.nan)
                for feature in feature_columns
                for stat in ["mean", "median", "std"]
            },
        }
        for (scope, cluster), group in cluster_data.groupby(["evaluation_scope", "cluster"], sort=True)
    ]
)
profile_long = pd.DataFrame(profile_rows)
representatives = pd.DataFrame(representative_rows)

profile_wide.to_csv(PATHS["profiles"] / "ho_so_cum_dbscan.csv", index=False)
profile_long.to_csv(PATHS["profiles"] / "ho_so_cum_dbscan_dang_dai.csv", index=False)
representatives.to_csv(PATHS["profiles"] / "quan_sat_dai_dien_cum.csv", index=False)

heatmap_source = cluster_data.loc[cluster_data["evaluation_scope"].eq(visual_scope)]
heatmap = (
    heatmap_source.groupby("cluster")[[f"{feature}_scaled" for feature in feature_columns]]
    .median()
)
heatmap.columns = [FEATURE_LABELS[feature] for feature in feature_columns]

fig, ax = plt.subplots(figsize=(14, max(3.5, 0.7 * len(heatmap))))
sns.heatmap(heatmap, cmap="vlag", center=0, annot=True, fmt=".2f", linewidths=0.4, ax=ax)
ax.set_title(f"Hồ sơ trung vị chuẩn hóa của cụm DBSCAN – {visual_scope}", fontweight="bold")
ax.set_xlabel("Chỉ tiêu")
ax.set_ylabel("Cụm DBSCAN")
fig.tight_layout()
profile_figure = PATHS["figures"] / "03_ho_so_cum_dbscan.png"
fig.savefig(profile_figure, dpi=180, bbox_inches="tight")
plt.show()
shutil.copy2(profile_figure, PATHS["report_figures"] / profile_figure.name)

display(profile_wide[["evaluation_scope", "cluster", "n_observations", "n_economies", "year_min", "year_max"]])
display(representatives.head(15))


## 12. Phân tích các điểm noise

Mỗi noise được so với phân phối non-noise trong đúng phạm vi DBSCAN bằng độ lệch tuyệt đối chuẩn hóa theo IQR. Ba đặc trưng lệch mạnh nhất được lưu để định hướng kiểm chứng. Đây chỉ là dấu hiệu thống kê: noise có thể đến từ cấu hình mật độ, thang đo, dữ liệu hoặc khác biệt kinh tế; không tự động đồng nghĩa khủng hoảng hay điều hành kém.


In [ ]:
noise_rows = []

for scope, scope_df in profile_data.groupby("evaluation_scope", sort=True):
    noise_scope = scope_df.loc[scope_df["cluster"].eq(-1)]
    reference = scope_df.loc[scope_df["cluster"].ne(-1)]
    if noise_scope.empty:
        continue
    if reference.empty:
        reference = scope_df

    scaled_cols = [f"{feature}_scaled" for feature in feature_columns]
    median = reference[scaled_cols].median()
    iqr = reference[scaled_cols].quantile(0.75) - reference[scaled_cols].quantile(0.25)
    iqr = iqr.replace(0, np.nan)

    for _, observation in noise_scope.iterrows():
        deviation = ((observation[scaled_cols] - median) / iqr).abs()
        deviation.index = feature_columns
        top = deviation.sort_values(ascending=False).head(3)
        row = {
            "evaluation_scope": scope,
            "country": observation["country"],
            "country_name": observation["country_name"],
            "year": int(observation["year"]),
            "cluster": -1,
            "point_type": observation["point_type"],
        }
        for feature in feature_columns:
            row[feature] = float(observation[f"{feature}_original"])
        for rank, (feature, value) in enumerate(top.items(), start=1):
            row[f"top_deviation_feature_{rank}"] = feature
            row[f"top_deviation_feature_name_{rank}"] = FEATURE_LABELS[feature]
            row[f"absolute_iqr_deviation_{rank}"] = float(value) if pd.notna(value) else np.nan
        row["interpretation_note"] = (
            "Khác biệt thống kê mạnh nhất ở các chỉ tiêu nêu trên; "
            "cần đối chiếu dữ liệu gốc, tham số DBSCAN và bối cảnh country-year trước khi kết luận."
        )
        noise_rows.append(row)

noise_output_columns = (
    ["evaluation_scope", "country", "country_name", "year", "cluster", "point_type"]
    + feature_columns
    + [
        item
        for rank in range(1, 4)
        for item in [
            f"top_deviation_feature_{rank}",
            f"top_deviation_feature_name_{rank}",
            f"absolute_iqr_deviation_{rank}",
        ]
    ]
    + ["interpretation_note"]
)
noise_analysis = pd.DataFrame(noise_rows, columns=noise_output_columns)
noise_path = PATHS["noise"] / "danh_sach_va_dac_trung_noise.csv"
noise_analysis.to_csv(noise_path, index=False)

if noise_analysis.empty:
    print("Cấu hình DBSCAN không tạo noise; đã xuất file CSV rỗng có chủ đích.")
else:
    noise_summary = (
        noise_analysis.groupby(["evaluation_scope", "top_deviation_feature_name_1"], as_index=False)
        .size()
        .rename(columns={"size": "n_noise"})
    )
    noise_summary.to_csv(PATHS["noise"] / "tong_hop_noise_theo_dac_trung.csv", index=False)

    plot_noise = (
        noise_analysis["top_deviation_feature_name_1"]
        .value_counts()
        .rename_axis("feature")
        .reset_index(name="n_noise")
    )
    fig, ax = plt.subplots(figsize=(11, 5.5))
    sns.barplot(data=plot_noise, y="feature", x="n_noise", color="#E45756", ax=ax)
    ax.set_title("Đặc trưng lệch mạnh nhất của các điểm noise", fontweight="bold")
    ax.set_xlabel("Số quan sát noise")
    ax.set_ylabel("")
    fig.tight_layout()
    noise_figure = PATHS["figures"] / "04_noise_theo_dac_trung_khac_biet.png"
    fig.savefig(noise_figure, dpi=180, bbox_inches="tight")
    plt.show()
    shutil.copy2(noise_figure, PATHS["report_figures"] / noise_figure.name)

    display(noise_analysis.head(15))
    print(f"Số noise: {len(noise_analysis)} ({len(noise_analysis) / len(profile_data):.1%} toàn mẫu)")


## 13. Nhận xét, giới hạn và checklist bàn giao

Cell cuối tự tổng hợp kết quả ở mức mô tả. Khi viết báo cáo cần luôn đặt điểm nội bộ cạnh độ phủ và phạm vi mẫu; không gọi một thuật toán là “tốt nhất” chỉ vì một chỉ số cao hơn. Hạn chế chính gồm: chỉ số nội bộ thiên về cụm lồi/tách biệt, PCA làm mất thông tin, panel có phụ thuộc thời gian trong cùng quốc gia, và nhãn noise nhạy với eps/min_samples.


In [ ]:
valid_main = comparison.query(
    "evaluation_set == 'algorithm_policy' and metric_note == 'Hợp lệ'"
).copy()

if valid_main.empty:
    print("Không có phép đánh giá chính hợp lệ; xem metric_note trong so_sanh_thuat_toan.csv.")
else:
    aggregate = (
        valid_main.groupby("algorithm", as_index=False)
        .agg(
            silhouette_mean=("silhouette", "mean"),
            davies_bouldin_mean=("davies_bouldin", "mean"),
            calinski_harabasz_mean=("calinski_harabasz", "mean"),
            coverage_mean=("coverage", "mean"),
            scopes=("evaluation_scope", "nunique"),
        )
    )
    silhouette_leader = aggregate.loc[aggregate["silhouette_mean"].idxmax()]
    dbscan_row = aggregate.loc[aggregate["algorithm"].eq("DBSCAN")]
    print(
        f"Silhouette trung bình cao nhất theo chính sách riêng: "
        f"{silhouette_leader['algorithm']} ({silhouette_leader['silhouette_mean']:.3f})."
    )
    if not dbscan_row.empty:
        dbscan_row = dbscan_row.iloc[0]
        print(
            f"DBSCAN: Silhouette TB={dbscan_row['silhouette_mean']:.3f}, "
            f"coverage TB={dbscan_row['coverage_mean']:.1%}. "
            "Hai con số phải được diễn giải đồng thời."
        )
    print(
        "Kết luận cuối cần kết hợp ba chỉ số, coverage, độ ổn định tham số và khả năng "
        "giải thích hồ sơ kinh tế; không suy luận nhân quả từ nhãn cụm/noise."
    )
    display(aggregate.round(4))

expected_outputs = [
    PATHS["evaluation"] / "so_sanh_thuat_toan.csv",
    PATHS["evaluation"] / "nhan_thuat_toan_doi_chung.csv",
    PATHS["evaluation"] / "lua_chon_mo_hinh_doi_chung.csv",
    PATHS["evaluation"] / "cau_hinh_thuat_toan_doi_chung.json",
    PATHS["evaluation"] / "pca_metadata.json",
    PATHS["profiles"] / "ho_so_cum_dbscan.csv",
    PATHS["profiles"] / "ho_so_cum_dbscan_dang_dai.csv",
    PATHS["profiles"] / "quan_sat_dai_dien_cum.csv",
    PATHS["noise"] / "danh_sach_va_dac_trung_noise.csv",
    PATHS["figures"] / "01_pca_so_sanh_thuat_toan.png",
    PATHS["figures"] / "02_so_sanh_chi_so_va_do_phu.png",
    PATHS["figures"] / "03_ho_so_cum_dbscan.png",
]
delivery_check = pd.DataFrame(
    {
        "output": [str(path.relative_to(ROOT)) for path in expected_outputs],
        "exists": [path.is_file() for path in expected_outputs],
        "bytes": [path.stat().st_size if path.is_file() else 0 for path in expected_outputs],
    }
)
display(delivery_check)
if not delivery_check["exists"].all():
    raise RuntimeError("Thiếu sản phẩm bàn giao; xem bảng delivery_check.")
print("Hoàn tất notebook 03: tất cả sản phẩm bắt buộc đã được tạo.")
